# BSAD and ML data sufficiency
## tl;dr
Real quote history supports a scoped BTC gross-risk study: 2,513 paired observations; 1,335 are OTM at entry across 72 sampled dates. Baselines and a small ML model were actually fitted. Twenty-two wide exits dominate broad-panel error, so blindly using all complete rows is unsafe. A separate full-session audit builds ten continuous 15-hour ledgers.

## Context & Methods
This diagnostic tests data sufficiency, not ML superiority or the final research specification. See `docs/RESEARCH_DATA_SUFFICIENCY.md` for source access and interpretation.
### Key Assumptions
Explicit sticky-moneyness (SM) benchmark; zero discounting; actual inverse-perpetual price formula with parallel percentage F/S/H basis assumption; gross BTC risk before funding, fees and execution. Midpoint IV and local OTM smile slope are separate inputs. All times use UTC.
The 72 dates are first-of-month sessions, October2020–September2026; only08–09 and16–17UTC are paired in that panel. They are not continuous six-year history. The OTM design was inspected after broad-panel results, so current OTM scores are exploratory. A formal comparison needs a frozen design and fresh holdout.


In [1]:
from pathlib import Path
import gzip
import json
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "option_collector").is_dir())
sys.path.insert(0, str(ROOT / "experiments"))
monthly = json.loads((ROOT / "docs/research_monthly_panel_metrics.json").read_text())
labels = json.loads((ROOT / "docs/research_label_quality_metrics.json").read_text())
daily = json.loads((ROOT / "docs/research_daily_quote_metrics.json").read_text())
rows = [json.loads(line) for line in gzip.open(ROOT / monthly["derived_rows_file"], "rt")]
print("Saved evidence loaded; no network or collector calls.")


Saved evidence loaded; no network or collector calls.


## Data
### 1. Count rows and independent time blocks
Cohorts were chosen from historical08UTC observations. Both exchange and receipt timestamps precede the decision. Central strikes are focal contracts; outside strikes provide smile support. Missing exits remain visible in the denominator.


In [2]:
print("Preselected focal observations:", monthly["counts"]["selected_focal_options"])
print("Paired rows:", monthly["rows"])
print("Dates / market intervals:", monthly["dates"], monthly["market_intervals"])
print("Usable pair share:", f"{monthly['rows'] / monthly['counts']['selected_focal_options']:.2%}")
print("OTM rows / dates / intervals:", labels["entry_otm"]["rows"], labels["entry_otm"]["dates"], labels["entry_otm"]["market_intervals"])


Preselected focal observations: 2718
Paired rows: 2513
Dates / market intervals: 72 144
Usable pair share: 92.46%
OTM rows / dates / intervals: 1335 72 144


## Results
### 2. Inspect label quality before fitting
A complete numeric row may still be economically misleading. Twenty-two broad-panel exits have spreads over50%; they account for most of the weighted BS error. Entry-OTM selection is causal and has no such wide exits in this sample. This does not make quote-band uncertainty disappear.


In [3]:
for name in ("all", "entry_otm", "entry_itm"):
    item = labels[name]
    print(name, "rows:", item["rows"], "wide exits:", item["wide_exits_above_50pct"])
    print("  BS squared-error share from wide exits:", f"{item['share_bs_squared_error_from_wide_exits']:.2%}")
    print("  Slope quote-band includes zero:", item["slope_band_contains_zero"])
print("Complete features missing values:", labels["entry_otm"]["feature_missing_values"])


all rows: 2513 wide exits: 22
  BS squared-error share from wide exits: 83.07%
  Slope quote-band includes zero: 1196
entry_otm rows: 1335 wide exits: 0
  BS squared-error share from wide exits: 0.00%
  Slope quote-band includes zero: 632
entry_itm rows: 1178 wide exits: 22
  BS squared-error share from wide exits: 84.01%
  Slope quote-band includes zero: 564
Complete features missing values: 0


### 3. Refit the small model from saved observations
The exact transformations and risk objective are in `research_panel_analysis.py` and `research_hedge_math.py`. This cell reruns chronological fitting from saved rows using NumPy. It does not fetch data. Scores remain exploratory because the design has been inspected; a null ML result is not a failed project.


In [4]:
from research_panel_analysis import FEATURES, train_and_assess
otm = [row for row in rows if bool(row["put"]) == (row["log_m"] < 0)]
assessment = train_and_assess(otm)
print("Features:", ", ".join(FEATURES))
print("Fit status:", assessment["status"])
for name, counts in assessment["partitions"].items():
    print(name, "rows / dates / intervals:", counts["rows"], counts["dates"], counts["market_intervals"])
print("Static calibrated adjustment coefficients:", assessment["hw_static_training_coefficients"])
print("Later-evaluation daily MSE (BTC squared):")
for method, loss in assessment["results"]["test"]["mean_daily_mse_btc_squared"].items():
    print(" ", method, f"{loss:.4g}")
print("ML versus SM day-bootstrap interval:", assessment["test_comparison_intervals"]["sm"]["day_bootstrap95_interval"])


Features: put, log_m, sqrt_tau, mid_iv, delta_bs, vega_coin, smile_slope, relative_spread, hedge_index_basis
Fit status: fitted_and_evaluated_chronologically
train rows / dates / intervals: 675 39 78
validation rows / dates / intervals: 240 12 24
test rows / dates / intervals: 420 21 42
Static calibrated adjustment coefficients: [-0.1411473024163436, -0.14954067987597855, 0.9363577369929357]
Later-evaluation daily MSE (BTC squared):
  unhedged 3.974e-06
  bs 2.074e-07
  sm 2.219e-07
  hw_static_fit 2.105e-07
  ml 2.287e-07
ML versus SM day-bootstrap interval: [-5.5137351761449005e-08, 4.791227323378118e-08]


### 4. Check continuous intraday construction
A separate1October2025 audit reads18options plus perpetual BBO and derivative ticker for the whole day. The research cohort is known at08UTC; only08–23UTC decisions are used. Ten focal contracts each have15 hourly outcomes. This is one15-hour session, not a24-hour overnight path or a multi-date dynamic-ML evaluation.


In [5]:
print("Files / raw CSV rows / compressed bytes:", daily["files_successful"], daily["raw_csv_rows"], daily["raw_gzip_bytes"])
print("Hourly states per complete contract:", len(daily["decision_hours_utc"]))
print("Focal pairs:", daily["pair_counts"]["sm_hedge_pairs"], "/", daily["potential_hourly_pairs"])
print("Complete 15-hour ledgers:", len(daily["gross_dynamic_15hour_ledgers"]))
ledger = daily["gross_dynamic_15hour_ledgers"][0]
print("Example instrument:", ledger["instrument"])
print("Observed option change (BTC):", ledger["option_change_coin"])
print("SM hedge cash flow / residual (BTC):", ledger["sm_hedge_gain_coin"], ledger["sm_gross_error_coin"])


Files / raw CSV rows / compressed bytes: 20 2951029 43921277
Hourly states per complete contract: 16
Focal pairs: 150 / 150
Complete 15-hour ledgers: 10
Example instrument: BTC-17OCT25-113000-C
Observed option change (BTC): 0.018250000000000002
SM hedge cash flow / residual (BTC): 0.0177061114411393 0.0005438885588607029


## Takeaways
- Data exist for fitting explicit adjusted-delta baselines and constructing nine features plus risk targets for a small model.
- The usable entry-OTM sample has1,335rows but72dates and144shared market intervals. More contracts are not more independent histories.
- Broad-panel label defects are material. The OTM experiment is an exploratory redesign, with no claim of ML superiority.
- A continuous within-day ledger works in one audited session. Everyday/overnight, ETH, order-flow/GARCH features and net execution need matching additional evidence.
- Use `docs/RESEARCH_DATA_SUFFICIENCY.md` and exact metric JSONs for access routes and remaining limitations. Raw market data remain in ignored local results.

Execution: cells executed sequentially in CPython3.12.14 with NumPy2.3.5. Jupyter/nbformat and plotting packages are absent; native kernel validation was not performed. This table-based diagnostic has no figures. Source and saved outputs are inspectable; rendering inspection is recorded in the report.
